# 🎓 Project — Student Management Tool

---

**Capstone project for Section 1: Python for Engineers.**

This project exercises everything you've learned so far:
data types, dictionaries, lists, functions, file I/O (JSON), exception handling,
input validation, type hints, docstrings, logging, and a bit of OOP.

## 📋 Overview

Build a small CLI-style tool that manages a list of students. The tool:

- **Stores** students in memory as Python dictionaries
- **Persists** them to disk as JSON (so data survives between runs)
- **Supports** add, view, update, delete, search, list-all
- **Validates** every input (no garbage data allowed)
- **Handles errors** gracefully (missing files, bad input, etc.)

By the end you'll have a working tool — not just snippets.

## ✅ Requirements

**Data model — each student is a dict:**
```python
{
    "id": int,
    "name": str,
    "age": int,
    "grade": str,
    "marks": {"math": int, "science": int, ...}
}
```

**Persistence:** all students live in `students.json` (created if missing).

**Functions to implement:**
- `add_student`, `view_student`, `update_student`, `delete_student`
- `list_all_students`, `search_by_name`, `calculate_average`

**Engineering practices:**
- Input validation (age 5–100, marks 0–100, name non-empty)
- Exception handling for bad input and missing files
- Type hints + docstrings everywhere
- Use `logging` instead of `print()` for internal events

---
## Step 1 — Data Model & File Helpers

First we set up our storage file and two helpers:
- `load_students()` — reads the JSON file, returns `[]` if it doesn't exist yet
- `save_students(students)` — writes the list back to disk

This is also where we configure `logging` so the rest of the notebook can use it.

In [ ]:
import json
import logging
import os

# Configure logging once for the whole notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("student_tool")

STUDENTS_FILE = "students.json"


def load_students() -> list[dict]:
    """Load all students from the JSON file. Returns [] if the file is missing."""
    if not os.path.exists(STUDENTS_FILE):
        log.info("No %s found — starting with an empty list.", STUDENTS_FILE)
        return []
    try:
        with open(STUDENTS_FILE, "r") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        log.error("Corrupt JSON in %s: %s — starting fresh.", STUDENTS_FILE, e)
        return []


def save_students(students: list[dict]) -> None:
    """Write the list of students to disk as pretty-printed JSON."""
    with open(STUDENTS_FILE, "w") as f:
        json.dump(students, f, indent=2)
    log.info("Saved %d student(s) to %s", len(students), STUDENTS_FILE)


In [ ]:
# Quick sanity check — load (file probably doesn't exist yet)
students = load_students()
print("Loaded:", students)

# Save an empty list to create the file
save_students(students)
print("Reloaded:", load_students())


---
## Step 2 — Validation Helpers

Garbage in, garbage out. Before we let anything reach our data store we validate it.
Each validator either **returns a clean value** or **raises `ValueError`** with a helpful message.

In [ ]:
def validate_name(name: str) -> str:
    """Strip whitespace and ensure the name is non-empty."""
    if not isinstance(name, str):
        raise ValueError(f"Name must be a string, got {type(name).__name__}")
    cleaned = name.strip()
    if not cleaned:
        raise ValueError("Name cannot be empty.")
    return cleaned


def validate_age(age: int) -> int:
    """Age must be an int between 5 and 100 inclusive."""
    if not isinstance(age, int) or isinstance(age, bool):
        raise ValueError(f"Age must be an int, got {type(age).__name__}")
    if age < 5 or age > 100:
        raise ValueError(f"Age must be between 5 and 100. Got: {age}")
    return age


def validate_marks(marks: dict) -> dict:
    """Each subject score must be an int between 0 and 100."""
    if not isinstance(marks, dict):
        raise ValueError("Marks must be a dict like {'math': 90, 'science': 85}")
    for subject, score in marks.items():
        if not isinstance(score, int) or isinstance(score, bool):
            raise ValueError(f"Score for {subject!r} must be int, got {type(score).__name__}")
        if score < 0 or score > 100:
            raise ValueError(f"Score for {subject!r} must be 0-100. Got: {score}")
    return marks


In [ ]:
# Try a few validators — good and bad input
print(validate_name("  Alice  "))         # -> 'Alice'
print(validate_age(22))                    # -> 22
print(validate_marks({"math": 90, "sci": 80}))

# Now break them on purpose
for bad in [("", validate_name), (200, validate_age), ({"math": 150}, validate_marks)]:
    value, fn = bad
    try:
        fn(value)
    except ValueError as e:
        print(f"{fn.__name__}({value!r}) -> {e}")


---
## Step 3 — CRUD Functions

Now we build the actual operations. Each one:
1. Loads the current list from disk
2. Validates input
3. Mutates the list
4. Saves it back

IDs are auto-assigned as `max(existing_id) + 1` (so they never collide).

In [ ]:
def _next_id(students: list[dict]) -> int:
    """Return the next available student id."""
    return max((s["id"] for s in students), default=0) + 1


def add_student(name: str, age: int, grade: str, marks: dict) -> dict:
    """Validate inputs, assign an id, save, and return the new student dict."""
    name = validate_name(name)
    age = validate_age(age)
    grade = validate_name(grade)   # reuse: grade is also a non-empty string
    marks = validate_marks(marks)

    students = load_students()
    student = {
        "id": _next_id(students),
        "name": name,
        "age": age,
        "grade": grade,
        "marks": marks,
    }
    students.append(student)
    save_students(students)
    log.info("Added student id=%d name=%s", student["id"], student["name"])
    return student


In [ ]:
def view_student(student_id: int) -> dict | None:
    """Return the student with the given id, or None if not found."""
    students = load_students()
    for s in students:
        if s["id"] == student_id:
            return s
    log.warning("No student found with id=%d", student_id)
    return None


def update_student(student_id: int, **kwargs) -> bool:
    """Update fields on a student. Returns True on success, False if id missing."""
    allowed = {"name", "age", "grade", "marks"}
    bad_keys = set(kwargs) - allowed
    if bad_keys:
        raise ValueError(f"Cannot update unknown field(s): {bad_keys}")

    students = load_students()
    for s in students:
        if s["id"] == student_id:
            if "name"  in kwargs: s["name"]  = validate_name(kwargs["name"])
            if "age"   in kwargs: s["age"]   = validate_age(kwargs["age"])
            if "grade" in kwargs: s["grade"] = validate_name(kwargs["grade"])
            if "marks" in kwargs: s["marks"] = validate_marks(kwargs["marks"])
            save_students(students)
            log.info("Updated student id=%d fields=%s", student_id, list(kwargs))
            return True
    log.warning("Update failed — no student with id=%d", student_id)
    return False


def delete_student(student_id: int) -> bool:
    """Delete a student by id. Returns True if deleted, False if not found."""
    students = load_students()
    new_list = [s for s in students if s["id"] != student_id]
    if len(new_list) == len(students):
        log.warning("Delete failed — no student with id=%d", student_id)
        return False
    save_students(new_list)
    log.info("Deleted student id=%d", student_id)
    return True


In [ ]:
def list_all_students() -> None:
    """Pretty-print every student as a small table."""
    students = load_students()
    if not students:
        print("(no students yet)")
        return
    print(f"{'ID':<4} {'Name':<20} {'Age':<5} {'Grade':<8} Marks")
    print("-" * 60)
    for s in students:
        print(f"{s['id']:<4} {s['name']:<20} {s['age']:<5} {s['grade']:<8} {s['marks']}")


def search_by_name(query: str) -> list[dict]:
    """Case-insensitive substring search on the student name."""
    q = query.strip().lower()
    if not q:
        return []
    return [s for s in load_students() if q in s["name"].lower()]


def calculate_average(student_id: int) -> float:
    """Return the mean of all subject scores for a student."""
    student = view_student(student_id)
    if student is None:
        raise ValueError(f"No student with id={student_id}")
    marks = student["marks"]
    if not marks:
        return 0.0
    return sum(marks.values()) / len(marks)


---
## Step 4 — Putting It Together (Usage Demo)

Let's actually use the tool: clear the file, add a few students, list them, search, update, delete, and compute an average.

In [ ]:
# Start fresh for the demo
save_students([])

add_student("Alice Johnson", 17, "A",  {"math": 92, "science": 88, "english": 95})
add_student("Bob Smith",     16, "B",  {"math": 75, "science": 80, "english": 70})
add_student("Charlie Diaz",  18, "A",  {"math": 88, "science": 91, "english": 84})
add_student("Diana Patel",   17, "C",  {"math": 60, "science": 65, "english": 70})

list_all_students()


In [ ]:
# Search
results = search_by_name("di")
print("Search 'di':")
for r in results:
    print(" ", r["name"])


In [ ]:
# Update Bob's marks
update_student(2, marks={"math": 85, "science": 82, "english": 78})
print(view_student(2))


In [ ]:
# Average for Alice (id=1)
print(f"Alice's average: {calculate_average(1):.2f}")


In [ ]:
# Delete Diana (id=4) and list again
delete_student(4)
list_all_students()


---
## Step 5 (Optional) — OOP Alternative Sketch

The dict-based version above is perfectly fine. But once a system grows, OOP tends to organise it better.
Here's a *sketch* of how this could be refactored — implement it fully in **Challenge 4** below.

```python
class Student:
    def __init__(self, name, age, grade, marks):
        self.name  = validate_name(name)
        self.age   = validate_age(age)
        self.grade = validate_name(grade)
        self.marks = validate_marks(marks)

    def average(self) -> float:
        return sum(self.marks.values()) / len(self.marks) if self.marks else 0.0

    def to_dict(self) -> dict:
        return {"name": self.name, "age": self.age,
                "grade": self.grade, "marks": self.marks}


class StudentManager:
    def __init__(self, path="students.json"):
        self.path = path
        self.students: list[Student] = []
        self.load()

    def add(self, student: Student) -> None: ...
    def find(self, sid: int) -> Student | None: ...
    def save(self) -> None: ...
    def load(self) -> None: ...
```

You'd get auto-validation in `__init__`, methods that belong to the data, and a clean separation between *one student* and *the collection*.

---
# 📝 Extension Challenges
---

The tool above is fully working. The challenges below extend it in ways real-world tools need.
Each challenge has a short description, expected behaviour, and a hint — then a stub cell for you to fill in.

## Challenge 1 — Top Scorers

**Problem:** Write `top_scorers(n: int) -> list[dict]` that returns the top `n` students sorted by their average marks (highest first).

**Expected behaviour:**
```python
top_scorers(2)
# -> [{'id': 1, 'name': 'Alice Johnson', ...}, {'id': 3, 'name': 'Charlie Diaz', ...}]
```

**💡 Hint:** `sorted(students, key=lambda s: sum(s['marks'].values())/len(s['marks']), reverse=True)[:n]`

In [ ]:
# Challenge 1 — Top Scorers
def top_scorers(n: int) -> list[dict]:
    pass

# print(top_scorers(2))


## Challenge 2 — CSV Export

**Problem:** Write `export_to_csv(path: str = "students.csv") -> None` that writes every student to a CSV file.

**Expected columns:** `id, name, age, grade, average` (flatten the marks dict into a single `average` column).

**💡 Hint:** Use the `csv` module:
```python
import csv
with open(path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[...])
    writer.writeheader()
    writer.writerow({...})
```

In [ ]:
# Challenge 2 — CSV Export
import csv

def export_to_csv(path: str = "students.csv") -> None:
    pass

# export_to_csv()


## Challenge 3 — Interactive CLI Menu

**Problem:** Build a `while True:` loop that shows a menu and dispatches to the right function.

```
=== Student Management ===
1. Add
2. View
3. Update
4. Delete
5. List all
6. Quit
Choice: _
```

**Requirements:**
- Wrap every `input()` in `try/except` so bad input doesn't crash the loop
- Use `int(input(...))` for ids and ages — catch `ValueError`
- Quit cleanly on choice `6`

**💡 Hint:** Skeleton:
```python
while True:
    choice = input("Choice: ").strip()
    if choice == "1": ...
    elif choice == "6": break
    else: print("Unknown choice")
```

*(Run this in a regular `.py` file or terminal — `input()` works in notebooks but blocks the kernel.)*

In [ ]:
# Challenge 3 — CLI Menu
def run_cli() -> None:
    while True:
        pass  # build the menu here

# run_cli()


## Challenge 4 — Full OOP Refactor

**Problem:** Take the sketch from Step 5 and turn it into a working implementation.

**Requirements:**
- `Student` class with `__init__`, `average()`, `to_dict()`, `__repr__`
- `StudentManager` class with `add`, `find`, `update`, `delete`, `list_all`, `save`, `load`
- Reuse the validators from Step 2 — don't duplicate logic
- The manager should behave identically to the function-based version above

**💡 Hint:** Build `Student` first, prove it works, then layer `StudentManager` on top.

In [ ]:
# Challenge 4 — OOP Refactor
class Student:
    def __init__(self, name: str, age: int, grade: str, marks: dict):
        pass

    def average(self) -> float:
        pass

    def to_dict(self) -> dict:
        pass


class StudentManager:
    def __init__(self, path: str = "students.json"):
        pass

    def add(self, student: Student) -> None:
        pass

    def find(self, sid: int) -> Student | None:
        pass

    def save(self) -> None:
        pass

    def load(self) -> None:
        pass


## 🚀 Bonus — Grade Distribution

**Problem:** Write `grade_distribution() -> dict[str, int]` that returns how many students have each grade.

**Expected output:**
```python
grade_distribution()
# -> {'A': 2, 'B': 1, 'C': 1}
```

**💡 Hint:** Use `collections.Counter`:
```python
from collections import Counter
return dict(Counter(s['grade'] for s in load_students()))
```

In [ ]:
# Bonus — Grade Distribution
from collections import Counter

def grade_distribution() -> dict[str, int]:
    pass

# print(grade_distribution())
